# AutoJev-style Qwen3.5-9B LoRA

Reproducible text-only pilot with AutoJev's 255-way learned decision readout, option-order augmentation, isolated calibration fold, and PEFT backbone. Select an A100 runtime before running.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available()
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())


## Install the pinned AutoJev environment

The upstream project pins its tested stack. Colab's preinstalled TorchAudio can target a different CUDA version, so this text-only experiment removes it.


In [ ]:
!rm -rf /content/autojev
!git clone --depth 1 https://github.com/denis-pplx/autojev.git /content/autojev
!python -m pip uninstall -y torchaudio torchao
!python -m pip install -q -e /content/autojev peft>=0.17
from pathlib import Path
p=Path('/content/autojev/src/autojev/data.py')
s=p.read_text().replace('\"maximum_input_tokens\": max(cast(int, row[\"source\"][\"input_tokens\"]) for row in rows),','\"maximum_input_tokens\": max((cast(int, row[\"source\"][\"input_tokens\"]) for row in rows), default=0),')
p.write_text(s)


If the preceding install asks for a runtime restart, restart once and continue here. The files under `/content` remain available.


In [ ]:
import torch, transformers, peft, autojev
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('transformers', transformers.__version__, 'peft', peft.__version__)


## Build the public text corpus

Half scale yields roughly 47,500 training rows. Evaluation and calibration stay separate by source family.


In [ ]:
!autojev-data --output /content/autojev-data --manifest /content/autojev-data-manifest.json --train-scale 0.5 --image-train 0 --image-eval 0 --max-length 8192
!cat /content/autojev-data-manifest.json | tail -40


## Write the LoRA + decision-readout trainer


In [ ]:
from pathlib import Path
TRAINER = '#!/usr/bin/env python3\n"""Train a Qwen3.5-9B LoRA with AutoJev\'s dedicated 255-way readout."""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport torch\nimport torch.nn.functional as F\nfrom peft import LoraConfig, get_peft_model\nfrom safetensors.torch import save_file\n\nfrom autojev.evaluate import fit_temperature, hard_label, label_index, metrics, options, read_rows, evaluate_logits\nfrom autojev.model import DecisionModel\n\nMODEL = "Qwen/Qwen3.5-9B"\nREVISION = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"\n\n\ndef shuffled(rows, rng):\n    result = copy.deepcopy(list(rows))\n    for row in result:\n        if row["question"]["type"] != "choice":\n            continue\n        criteria, target = row["question"]["criteria"], row["target"]\n        soft = dict(zip(criteria, target, strict=True)) if isinstance(target, list) else None\n        items = list(criteria.items())\n        rng.shuffle(items)\n        row["question"]["criteria"] = dict(items)\n        if soft is not None:\n            row["target"] = [soft[key] for key, _ in items]\n    return result\n\n\ndef target_tensor(rows, device):\n    values = torch.zeros((len(rows), 255), dtype=torch.float32, device=device)\n    for i, row in enumerate(rows):\n        labels, target = options(row["question"]), row["target"]\n        if isinstance(target, list):\n            distribution = target\n        elif row["question"]["type"] == "noul":\n            distribution = [1.0 - float(target), float(target)]\n        else:\n            distribution = [float(label == target) for label in labels]\n        values[i, :len(distribution)] = torch.tensor(distribution, device=device)\n    return values\n\n\n@torch.inference_mode()\ndef infer(model, rows, batch_size, max_length):\n    model.eval()\n    all_logits = []\n    for start in range(0, len(rows), batch_size):\n        batch_rows = rows[start:start + batch_size]\n        batch = model.prepare(batch_rows, max_length=max_length)\n        logits = model(batch).cpu()\n        all_logits.extend(logits[i, :len(options(row["question"]))].tolist() for i, row in enumerate(batch_rows))\n    model.train()\n    return all_logits\n\n\ndef evaluate(model, dev, calibration, batch_size, max_length):\n    cal_logits = infer(model, calibration, batch_size, max_length)\n    temperature = fit_temperature(cal_logits, [label_index(options(r["question"]), hard_label(r)) for r in calibration])\n    dev_logits = infer(model, dev, batch_size, max_length)\n    return temperature, metrics(evaluate_logits(dev, dev_logits, temperature))\n\n\ndef save_artifact(model, out, temperature, config):\n    out.mkdir(parents=True, exist_ok=True)\n    model.backbone.save_pretrained(out / "adapter")\n    save_file({"weight": model.readout.weight.detach().cpu().contiguous()}, out / "readout.safetensors")\n    model.processor.save_pretrained(out / "processor")\n    config = {**config, "temperature": temperature, "codes": model.codes, "token_ids": model.token_ids}\n    (out / "decision_config.json").write_text(json.dumps(config, indent=2) + "\\n")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", type=Path, default=Path("/content/autojev-data"))\n    ap.add_argument("--out", type=Path, default=Path("/content/autojev-9b-lora"))\n    ap.add_argument("--steps", type=int, default=20)\n    ap.add_argument("--max-length", type=int, default=2048)\n    ap.add_argument("--grad-accum", type=int, default=16)\n    ap.add_argument("--micro-batch", type=int, default=1)\n    ap.add_argument("--eval-rows", type=int, default=128)\n    ap.add_argument("--seed", type=int, default=20260922)\n    ap.add_argument("--lr", type=float, default=5e-5)\n    args = ap.parse_args()\n    args.out.mkdir(parents=True, exist_ok=True)\n    random.seed(args.seed); torch.manual_seed(args.seed)\n    train = read_rows(args.data / "train.jsonl")\n    dev = read_rows(args.data / "dev.jsonl")[:args.eval_rows]\n    calibration = read_rows(args.data / "temperature.jsonl")[:args.eval_rows]\n    rng = random.Random(args.seed)\n    rng.shuffle(train)\n\n    model = DecisionModel(train=True, base_model=MODEL, revision=REVISION, gradient_checkpointing=True, cpu_threads=8)\n    model.backbone = get_peft_model(model.backbone, LoraConfig(\n        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n    ))\n    model.readout.requires_grad_(True)\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    optimizer = torch.optim.AdamW(trainable, lr=args.lr, weight_decay=0.01)\n    config = {\n        "format_version": 1, "architecture": "autojev-readout+lora", "base_model": MODEL,\n        "revision": REVISION, "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05,\n        "max_length": args.max_length, "micro_batch": args.micro_batch, "grad_accum": args.grad_accum, "seed": args.seed,\n        "train_rows_available": len(train), "dev_rows": len(dev), "temperature_rows": len(calibration),\n        "trainable_parameters": sum(p.numel() for p in trainable),\n    }\n    (args.out / "run_config.json").write_text(json.dumps(config, indent=2) + "\\n")\n    print(json.dumps(config), flush=True)\n    torch.cuda.reset_peak_memory_stats()\n    started = time.monotonic()\n    log = []\n    optimizer.zero_grad(set_to_none=True)\n    for step in range(1, args.steps + 1):\n        losses = []\n        for micro in range(args.grad_accum):\n            first = ((step - 1) * args.grad_accum + micro) * args.micro_batch\n            rows = shuffled([train[(first + j) % len(train)] for j in range(args.micro_batch)], rng)\n            batch = model.prepare(rows, max_length=args.max_length)\n            logits = model(batch)\n            target = target_tensor(rows, logits.device)\n            loss = -(target * F.log_softmax(logits, dim=-1)).sum(-1).mean()\n            (loss / args.grad_accum).backward()\n            losses.append(float(loss.detach()))\n        torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n        optimizer.step(); optimizer.zero_grad(set_to_none=True)\n        entry = {"step": step, "loss": sum(losses) / len(losses), "allocated_gib": torch.cuda.memory_allocated() / 2**30}\n        log.append(entry); print(json.dumps(entry), flush=True)\n    temperature, dev_metrics = evaluate(model, dev, calibration, 1, args.max_length)\n    report = {\n        "status": "complete", "steps": args.steps, "examples_seen": args.steps * args.grad_accum * args.micro_batch,\n        "temperature": temperature, "development": dev_metrics,\n        "wall_seconds": time.monotonic() - started,\n        "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,\n        "last_loss": log[-1]["loss"],\n    }\n    save_artifact(model, args.out / "selected", temperature, {**config, **report})\n    (args.out / "training.jsonl").write_text("".join(json.dumps(x) + "\\n" for x in log))\n    (args.out / "report.json").write_text(json.dumps(report, indent=2) + "\\n")\n    print(json.dumps(report, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
Path('/content/autojev_9b_lora_train.py').write_text(TRAINER)


## Memory preflight

Twenty steps validate loading, gradients, the decision head, calibration, and artifact saving. Training uses 2,048 tokens initially while the saved serving contract supports the upstream 8,192-token limit.


In [ ]:
!python /content/autojev_9b_lora_train.py --steps 20 --micro-batch 1 --grad-accum 8 --eval-rows 64 --out /content/autojev-9b-lora-preflight


In [ ]:
import json
print(json.dumps(json.load(open('/content/autojev-9b-lora-preflight/report.json')), indent=2))


## Full pilot

Run only after the preflight succeeds. At 50k rows and effective batch 32 this is about 1,500 optimizer steps. Frequent checkpoints should be added before an unattended Colab run.


In [ ]:
# Example full-pilot command; intentionally commented until the largest safe microbatch is measured.
# !python /content/autojev_9b_lora_train.py --steps 1485 --micro-batch 4 --grad-accum 2 --eval-rows 512 --out /content/autojev-9b-lora-full


## Download the artifact


In [ ]:
!tar -C /content -czf /content/autojev-9b-lora-preflight.tar.gz autojev-9b-lora-preflight
from google.colab import files
# files.download('/content/autojev-9b-lora-preflight.tar.gz')
